# Would a weekly balance check catch the cash shortfall?

A supplier can finish the month with money in the bank and still have too little cash for payroll along the way. This lab lets you change which day the bookkeeper checks the balance and see which shortfalls appear on the report.

Start with the late-customer case. The customer pays A$15,000 on day 15. Before running the cells, predict what a bookkeeper who checks on days 1, 8, 15, 22 and 29 would report as the lowest balance. Then compare that figure with the daily ledger.

The three cases use invented Australian-dollar amounts and 30-day months. The notebook works offline with standard Python 3. Each code cell includes its saved result for readers without Python.

In [1]:
cases = {'late_customer': {'opening': 5000,
                   'adjustable': 'customer_invoice',
                   'receipts': [('customer_invoice', 10, 15000)],
                   'payments': [(7, 6000), (14, 4000), (21, 2000)]},
 'growing_order': {'opening': 5000,
                   'adjustable': 'new_order',
                   'receipts': [('ordinary_customer', 10, 10000),
                                ('new_order', 28, 20000)],
                   'payments': [(3, 12000), (12, 4000), (15, 8000)]},
 'lumpy_costs': {'opening': 6000,
                 'adjustable': 'project_invoice',
                 'receipts': [('sales_first', 10, 5000),
                              ('sales_second', 20, 5000),
                              ('project_invoice', 24, 10000)],
                 'payments': [(1, 4000),
                              (4, 6000),
                              (7, 2000),
                              (14, 2000),
                              (21, 2000),
                              (28, 2000)]}}

def balances(case_name, delay_days=0):
    case = cases[case_name]
    changes = [0] * 31
    for label, due_day, amount in case["receipts"]:
        day = due_day + (delay_days if label == case["adjustable"] else 0)
        if 1 <= day <= 30:
            changes[day] += amount
    for day, amount in case["payments"]:
        changes[day] -= amount
    ledger = [case["opening"]]
    for change in changes[1:]:
        ledger.append(ledger[-1] + change)
    return ledger

def report(ledger, inspection_days):
    daily_low = min(ledger)
    reported_low = min(ledger[day] for day in inspection_days)
    cash_gap = max(0, -daily_low)
    reported_gap = max(0, -reported_low)
    return daily_low, reported_low, cash_gap - reported_gap

## Change the dates

The default delay moves the customer receipt from day 10 to day 15. Change `first_check` to any value from 1 to 7 to shift the weekly reporting calendar. Every report includes opening and closing cash. A negative daily balance means a payment is unfunded. The model assumes no overdraft.

In [2]:
case_name = "late_customer"
delay_days = 5
first_check = 1

assert case_name in cases and 0 <= delay_days <= 21 and 1 <= first_check <= 7
ledger = balances(case_name, delay_days)
checked = sorted({0, 30, *range(first_check, 31, 7)})
daily_low, weekly_low, missed_gap = report(ledger, checked)
print("Days checked:", checked)
print(f"Closing cash: A${ledger[-1]:,}")
print(f"Lowest daily balance: {daily_low:,} AUD on day {ledger.index(daily_low)}")
print(f"Lowest reported balance: {weekly_low:,} AUD")
print(f"Cash shortfall missing from this report: A${missed_gap:,}")

Days checked: [0, 1, 8, 15, 22, 29, 30]
Closing cash: A$8,000
Lowest daily balance: -5,000 AUD on day 14
Lowest reported balance: -1,000 AUD
Cash shortfall missing from this report: A$4,000


## Compare all seven weekly schedules

The list runs through the seven possible start days for weekly checks. It counts the number of schedules that understate the daily funding gap for each delay. These counts describe seven calendars given equal space in an exercise. They say nothing about how often Australian businesses run short of cash.

In [3]:
for delay in (0, 4, 5, 15):
    path = balances("late_customer", delay)
    missed = [report(path, {0, 30, *range(start, 31, 7)})[2] for start in range(1, 8)]
    print(f"Receipt day {10 + delay:2}: {sum(x > 0 for x in missed)}/7 schedules "
          f"understate the gap. Largest understatement: A${max(missed):,}.")

Receipt day 10: 4/7 schedules understate the gap. Largest understatement: A$1,000.
Receipt day 14: 0/7 schedules understate the gap. Largest understatement: A$0.
Receipt day 15: 6/7 schedules understate the gap. Largest understatement: A$4,000.
Receipt day 25: 3/7 schedules understate the gap. Largest understatement: A$2,000.


## Check the ledger

The two calendars below have the same opening and closing balances. Only the day-14 balance changes when the receipt moves back one day. In the growing-order case the day-3 materials payment alone exceeds opening cash. Check when the largest shortfall actually occurs.

In [4]:
day14, day15 = (balances("late_customer", delay) for delay in (4, 5))
assert day14[-1] == day15[-1] == 8000
assert (min(day14), min(day15)) == (-1000, -5000)
assert [d for d in range(31) if day14[d] != day15[d]] == [14]
assert min(balances("growing_order")) == -9000
assert min(balances("lumpy_costs")) == -6000
print("All cross-checks agree with the published Cash-flow Cliff cases.")

All cross-checks agree with the published Cash-flow Cliff cases.


### Notes for teachers

Ask learners to find a weekly schedule that catches both the day-7 and day-14 lows. To explore a different business, change the case to `growing_order` or `lumpy_costs`. The daily ledger nets all movements on the same date. A supplier might need more cash within that day. Payments outside the 30-day window are excluded.

Joe, an AI assistant at Finofin, prepared this lab for SMB Loan. SMB Loan is an Australian business finance information publisher and may receive referral income from Lend. All amounts and dates here are fictional. The publisher’s [industry working-capital study](https://smbloan.com.au/research/working-capital-by-industry) uses a separate method to model operating expenses. The [Cash-flow Cliff timing study and data](https://thisismyuserdealwithit.github.io/smbloan-business-survival-notes/cash-flow-cliff/timing-study.html) give the source cases and deposit thresholds. See the [reuse terms](https://thisismyuserdealwithit.github.io/smbloan-business-survival-notes/cash-flow-cliff/LICENSE.txt) for the program, documentation and invented data.